# Explore → BenchPress → Build

This notebook is our workbench for the Kaggle "Measuring Progress Toward AGI" competition
(Executive Functions track: planning, inhibitory control, cognitive flexibility).

The workflow is:
1. **Design** benchmark tasks using the `kaggle-benchmarks` SDK
2. **Run** them on multiple LLMs to get scores
3. **Check novelty** with BenchPress — does our benchmark measure something *new*,
   or can its scores be predicted from existing benchmarks?
4. **Iterate** — if predictable, the benchmark is redundant; if novel *and* meaningful, we have a winner.

---

**Part A** — Learn the kaggle-benchmarks SDK by running existing examples  
**Part B** — Load the BenchPress matrix and build a `check_novelty()` function  
**Part C** — Multi-model runner + first Executive Functions task

---
## Part A: The kaggle-benchmarks SDK

The SDK gives us three core primitives:

- **`@task`** — decorator that wraps a function into a benchmark task. Each task gets
  an `llm` handle and returns a score (bool, float, or structured).
- **`llm.prompt()`** — sends a prompt to the model. Can request structured output via `schema=`.
- **`assertions.*`** — checks on the response (assert_in, assert_not_empty, etc.).
  These get recorded on the run object for inspection.

The SDK talks to models through an OpenAI-compatible API. We configure which provider/model
to use via environment variables: `MODEL_PROXY_URL`, `MODEL_PROXY_API_KEY`, `LLM_DEFAULT`.

Under the hood, `from kaggle_benchmarks import llm` immediately creates a model handle
using those env vars — so they must be set *before* the import.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# ── Provider registry ──
# Each provider is: (base_url, env_var_for_api_key, cheap_default_model)
#
# The SDK uses the OpenAI client under the hood, so any provider that speaks
# the OpenAI chat/completions API works. Gemini and xAI both have compat endpoints.
#
# NOTE: Anthropic is listed but currently out of credits. Swap back in when refilled.
PROVIDERS = {
    'openai': ('https://api.openai.com/v1',                                'OPENAI_API_KEY', 'gpt-4.1-mini'),
    'gemini': ('https://generativelanguage.googleapis.com/v1beta/openai',   'GEMINI_API_KEY', 'gemini-2.5-flash'),
    'xai':    ('https://api.x.ai/v1',                                      'XAI_API_KEY',    'grok-3-beta'),
    # 'anthropic': ('https://api.anthropic.com/v1', 'ANTHROPIC_API_KEY', 'claude-sonnet-4-20250514'),
}

# ── Set defaults for the SDK import ──
# The SDK reads these at import time to create the global `llm` object.
# We default to OpenAI/gpt-4.1-mini — cheapest option for exploration.
os.environ['MODEL_PROXY_URL']     = PROVIDERS['openai'][0]
os.environ['MODEL_PROXY_API_KEY'] = os.environ[PROVIDERS['openai'][1]]
os.environ['LLM_DEFAULT']         = PROVIDERS['openai'][2]

print(f'✓ Default provider: openai / {os.environ["LLM_DEFAULT"]}')
print(f'  Available providers: {list(PROVIDERS.keys())}')

✓ Default provider: openai / gpt-4.1-mini
  Available providers: ['openai', 'gemini', 'xai']


In [ ]:
# Now we can safely import. The SDK creates a global `llm` handle pointing at
# our default model. We also grab `task`, `assertions`, and the lower-level
# `ModelProxy` for creating handles to other providers later.

from kaggle_benchmarks import assertions, llm, task
from kaggle_benchmarks.kaggle.model_proxy import ModelProxy

print(f'✓ SDK loaded')
print(f'  Default llm: {llm.name}')  # e.g. 'gpt-4.1-mini'

✓ SDK loaded
  Default llm: gpt-4.1-mini


In [ ]:
# ── Model factory ──
# ModelProxy creates an LLM handle given (model_name, api_key, base_url).
# It returns an OpenAI-client wrapper that task.run() can call.
#
# GOTCHA: The SDK sends `seed=0` in every API call for reproducibility.
# OpenAI accepts this, but Gemini and xAI reject it (400 error).
# Workaround: we patch `invoke()` to strip the seed parameter.
#
# Also: `load_model()` from the SDK only reads env vars — it doesn't accept
# api_key/base_url as arguments. So we use ModelProxy directly.

def make_model(model_name, api_key, base_url, needs_seed_patch=False):
    """Create an LLM handle. Optionally patch out `seed` for providers that reject it."""
    m = ModelProxy(model=model_name, api_key=api_key, base_url=base_url)
    if needs_seed_patch:
        orig = m.invoke
        def patched(messages, system=None, **kw):
            kw.pop('seed', None)
            return orig(messages, system=system, **kw)
        m.invoke = patched
    return m


def get_llm(provider, model=None):
    """Get an LLM handle for a given provider. Uses default model if none specified."""
    base_url, key_env, default_model = PROVIDERS[provider]
    needs_patch = provider in ('gemini', 'xai')  # these reject seed=0
    return make_model(model or default_model, os.environ[key_env], base_url, needs_patch)


# Quick smoke test — does our factory work?
test_llm = get_llm('openai')
print(f'✓ Factory works: {test_llm.name}')

✓ Factory works: gpt-4.1-mini


### A1: simple_task — hello world

The simplest benchmark pattern: a **parent task** that runs several **subtasks**
and returns the fraction that passed.

Key SDK concepts shown here:
- `@task(name, description=)` — registers a function as a benchmark task
- `llm.prompt(text)` — sends prompt, returns string response
- `assertions.assert_in(needle, haystack)` — records a pass/fail check
- `subtask.run(llm)` — executes a subtask, returns a `Run` object
- `run.passed` — did all assertions pass?
- `run.result` — the return value of the task function

In [ ]:
# Two subtasks: one just checks the model responds, the other checks it can
# extract a name from the prompt.

@task("subtask1", description="greeting")
def subtask1(llm):
    response = llm.prompt("Hello!")
    assertions.assert_not_empty(response)  # just check it said *something*

@task("subtask2", description="introduction")
def subtask2(llm, name: str):
    response = llm.prompt(f"My name is {name}. What is my name?")
    assertions.assert_in(name, response)   # check the name appears in the response


# Parent task: runs subtasks and scores as fraction passed.
# Note the return type hint `-> float` — this tells the SDK the result is a score,
# not just pass/fail.

@task("simple_task", description="top level task")
def simple_task(llm) -> float:
    subtask_runs = [
        subtask1.run(llm),
        subtask2.run(llm, "Alan Turing"),
        subtask2.run(llm, "Richard Feynman"),
    ]
    return sum(s.passed for s in subtask_runs) / len(subtask_runs)


run = simple_task.run(llm)
print(f'Score: {run.result}, Passed: {run.passed}')

Score: 1.0, Passed: True


In [ ]:
# ── Anatomy of a Run object ──
# Every task.run() returns a Run with:
#   .result      — the task's return value (float score, bool, None for void tasks)
#   .passed      — True if all assertions passed AND no exceptions
#   .status      — 'success' or 'error'
#   .subruns     — a Runs container with child runs (for parent tasks)
#   .chat        — the conversation history (prompts + responses + assertion results)
#   .assertion_results — list of AssertionResult objects

print(f'Result:  {run.result}')
print(f'Passed:  {run.passed}')
print(f'Status:  {run.status}')
print(f'Subruns: {len(run.subruns.runs)}')
print()
for i, sr in enumerate(run.subruns.runs):
    print(f'  [{i}] {sr.name}: passed={sr.passed}, result={sr.result}')

Result:  1.0
Passed:  True
Status:  success
Subruns: 3

  [0] llm=gpt-4.1-mini/Run #1: passed=True, result=None
  [1] llm=gpt-4.1-mini, name=Alan Turing/Run #1: passed=True, result=None
  [2] llm=gpt-4.1-mini, name=Richard Feynman/Run #2: passed=True, result=None


### A2: monty_hall — structured output

The SDK can request **structured output** from the model via `llm.prompt(text, schema=...)`:
- `schema=bool` → model returns True/False
- `schema={"reasoning": str, "answer": Literal["A", "B"]}` → model returns an object
  with `.reasoning` and `.answer` fields

This is powered by OpenAI's structured output / JSON mode under the hood.

Below we test several Monty Hall variants — classic probability reasoning.

In [ ]:
from typing import Literal

# Classic Monty Hall: schema=bool means the model must return True or False.
# Correct answer: True (you should switch).
@task()
def monty_hall(llm):
    return llm.prompt(
        """Suppose you're on a game show. You're given the choice of three doors:
Behind one door is a car; behind the others, goats.
You pick a door, say No. 1. The host, who knows what's behind the doors,
opens another door, say No. 3, revealing a goat.
He then says to you, \"Do you want to switch to door No. 2?\"
Is it to your advantage to switch your choice?""",
        schema=bool,
    )

# Inverted: 2 cars 1 goat + switching costs $1000. Answer: don't switch.
@task()
def monty_hall_inverted(llm):
    answer = llm.prompt(
        """Suppose you're on a game show. You're given the choice of three doors:
Behind one door is a goat; behind the others, cars.
You pick a door, say No. 1. The host, who knows what's behind the doors,
opens another door, say No. 3, revealing a goat.
He then says to you, \"You can pay $1000 to switch to door No. 2?\"
Is it to your advantage to switch your choice?""",
        schema=bool,
    )
    return not answer  # correct if model says False (don't switch)

# Rephrased as escape room. Uses structured output with reasoning + answer.
# The `schema` dict creates a Pydantic-like object: response.reasoning, response.answer
@task()
def rephrased_monty_hall(llm):
    response = llm.prompt(
        """You're in an escape room with three locked doors. Only one leads to the exit.
You choose the first door. Before you try the key, the game master says:
\"The second door is definitely wrong.\"

A: Stick with your original choice (door one).
B: Switch to the third door.

What gives you the best chance of escaping?""",
        schema={"reasoning": str, "answer": Literal["A", "B"]},
    )
    return response.answer.lower() == "b"  # switching is correct


for t in [monty_hall, monty_hall_inverted, rephrased_monty_hall]:
    r = t.run(llm)
    print(f'{t.name}: result={r.result}, passed={r.passed}')

Wrong return type <class 'bool'>. Expected None | kaggle_benchmarks.results.Unknown. This may need to lead to unexpected task behavior.


Monty Hall: result=True, passed=False


Wrong return type <class 'bool'>. Expected None | kaggle_benchmarks.results.Unknown. This may need to lead to unexpected task behavior.


Monty Hall Inverted: result=True, passed=False


Wrong return type <class 'bool'>. Expected None | kaggle_benchmarks.results.Unknown. This may need to lead to unexpected task behavior.


Rephrased Monty Hall: result=True, passed=False


### A3: Run same tasks on a different model

Here's where our `get_llm()` factory pays off — we can create a handle to
any provider and pass it to `task.run()`.

In [ ]:
# Create a Gemini handle. Uses our factory which patches out the `seed` param.
gemini = get_llm('gemini')
print(f'Testing with: {gemini.name}\n')

for t in [monty_hall, monty_hall_inverted, rephrased_monty_hall]:
    r = t.run(gemini)
    print(f'  {t.name}: result={r.result}, passed={r.passed}')

Testing with: gemini-2.5-flash



Wrong return type <class 'bool'>. Expected None | kaggle_benchmarks.results.Unknown. This may need to lead to unexpected task behavior.


  Monty Hall: result=True, passed=False


Wrong return type <class 'bool'>. Expected None | kaggle_benchmarks.results.Unknown. This may need to lead to unexpected task behavior.


  Monty Hall Inverted: result=True, passed=False


Wrong return type <class 'bool'>. Expected None | kaggle_benchmarks.results.Unknown. This may need to lead to unexpected task behavior.


  Rephrased Monty Hall: result=True, passed=False
